In [0]:
%sql
-- Criando tabela fato economic

CREATE OR REPLACE TABLE portfolio_energia.gold.fact_economic AS

WITH world_bank_granularidade AS (
    SELECT 
        dl.id_location AS location_id,
        dd.date_id AS date_id,
        di.indicator_id AS indicator_id,
        wb.metric AS metric
   FROM (
        SELECT date, iso3, indicator_code, metric FROM portfolio_energia.silver.world_bank_silver_populacao 
        UNION
        SELECT date, iso3, indicator_code, metric FROM portfolio_energia.silver.world_bank_silver_pfdi
        UNION
        SELECT date, iso3, indicator_code, metric  FROM portfolio_energia.silver.world_bank_silver_peletricidade
        UNION
        SELECT date, iso3, indicator_code, metric FROM portfolio_energia.silver.world_bank_silver_pib_per_capita
   ) AS wb
        LEFT JOIN portfolio_energia.gold.dim_location AS dl
            ON wb.iso3 = dl.iso3
        LEFT JOIN portfolio_energia.gold.dim_date AS dd
            ON CAST(date_format(wb.date, 'yyyyMMdd') AS INT)  = dd.date_id
        LEFT JOIN portfolio_energia.gold.dim_indicator AS di
            ON wb.indicator_code = di.indicator_code
    )
SELECT
    md5(concat(CAST(location_id AS STRING), CAST(date_id AS STRING), CAST(indicator_id AS STRING))) AS economic_id,
    location_id,
    date_id,
    indicator_id,
    metric
FROM 
    world_bank_granularidade;

-- criando tabela fato carbon_emission

CREATE OR REPLACE TABLE portfolio_energia.gold.fact_carbon_emission AS
SELECT
    md5(concat(CAST(dl.id_location AS STRING), CAST(dd.date_id AS STRING))) AS emission_id,
    dl.id_location AS id_location,
    dd.date_id,
    es.value AS emission 
FROM 
    portfolio_energia.silver.ember_intensity_silver AS es
LEFT JOIN portfolio_energia.gold.dim_location AS dl
    ON es.iso3 = dl.iso3
LEFT JOIN portfolio_energia.gold.dim_date AS dd
    ON CAST(date_format(es.date, 'yyyyMMdd') AS INT) = dd.date_id;

--criando tabela fact_eletricity

CREATE OR REPLACE TABLE portfolio_energia.gold.fact_electricity AS
SELECT
    md5(concat(CAST(dl.id_location AS STRING), CAST(dd.date_id AS STRING))) AS demand_id,
    dl.id_location AS id_location,
    dd.date_id,
    ed.value AS demand 
FROM 
    portfolio_energia.silver.ember_demand_silver AS ed
LEFT JOIN portfolio_energia.gold.dim_location AS dl
    ON ed.iso3 = dl.iso3 
LEFT JOIN portfolio_energia.gold.dim_date AS dd
    ON CAST(date_format(ed.date, 'yyyyMMdd') AS INT) = dd.date_id;